# Week 3: End-to-End 파이프라인 - 이미지 → 데이터 자동화

Week 1~2에서 만든 기능을 하나의 파이프라인으로 연결하고,
여러 EMR 스캔 파일을 **일괄 처리하여 CSV로 내보내는** 자동화를 만듭니다.

## 파이프라인
```
EMR 이미지들 → GPT-5.4-nano (텍스트 추출) → GPT-5.4-nano (JSON 구조화) → DataFrame → CSV 다운로드
```

## 이번 주 목표
- Week 1+2를 하나의 함수로 통합
- 폴더 단위 일괄 처리
- 결과를 DataFrame으로 변환하여 CSV/Excel로 내보내기

---

## 1. 환경 설정

In [ ]:
!pip install -q openai pillow pandas openpyxl

import os
import json
import base64
import glob
from getpass import getpass
from openai import OpenAI
import pandas as pd

api_key = getpass("OpenAI API Key를 입력하세요: ")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

print("설정 완료!")

## 2. 샘플 데이터 준비

In [ ]:
# GitHub에서 샘플 EMR 이미지 다운로드
!git clone --depth 1 https://github.com/Yuri-daepo/hospital-emr-ocr.git /content/repo 2>/dev/null || true
!cp /content/repo/emr_samples/*.png /content/emr_input/ 2>/dev/null || true
!mkdir -p /content/emr_input
!cp /content/repo/emr_samples/*.png /content/emr_input/ 2>/dev/null || true

INPUT_DIR = "/content/emr_input"
samples = sorted(glob.glob(f"{INPUT_DIR}/*.png"))
print(f"처리할 EMR 이미지: {len(samples)}개")
for f in samples:
    print(f"  - {os.path.basename(f)}")

## 3. 통합 파이프라인 함수

Week 1(이미지→텍스트)과 Week 2(텍스트→JSON)를 하나로 합칩니다.

In [ ]:
# --- 설정 ---
EXTRACTION_FIELDS = {
    "hospital_name": "병원명",
    "patient_name": "환자명",
    "patient_id": "환자번호",
    "birth_date": "생년월일 (YYYY-MM-DD)",
    "gender": "성별 (M/F)",
    "visit_date": "진료일 (YYYY-MM-DD)",
    "department": "진료과",
    "doctor_name": "담당의",
    "diagnosis": "진단명 (콤마 구분 문자열)",
    "chief_complaint": "주소/주증상",
    "vital_signs": "활력징후 (BP/HR/BT 등을 하나의 문자열로)",
    "medications": "처방 약물 (약물명+용량+용법을 콤마 구분 문자열로)",
    "notes": "기타 소견",
}

fields_description = "\n".join(
    f"- {k}: {v}" for k, v in EXTRACTION_FIELDS.items()
)

STRUCTURING_PROMPT = f"""당신은 의료 문서 데이터 추출 전문가입니다.

아래 EMR 텍스트에서 다음 필드를 추출하여 JSON으로 구조화하세요.

## 추출할 필드
{fields_description}

## 규칙
1. 문서에 없는 필드는 null
2. 배열이 아닌 콤마 구분 문자열로 통일 (CSV 변환 용이)
3. 날짜는 YYYY-MM-DD 형식
4. JSON만 출력

## 텍스트
{{text}}
"""


# --- 핵심 함수 ---
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def process_single_image(image_path):
    """이미지 1장 → 구조화된 dict 반환"""
    b64 = encode_image(image_path)

    # Step 1: 이미지 → 텍스트
    resp1 = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "이 의료 문서 이미지에서 모든 텍스트를 정확하게 추출해주세요."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}", "detail": "high"}}
            ]
        }],
        max_tokens=4096,
    )
    raw_text = resp1.choices[0].message.content

    # Step 2: 텍스트 → JSON
    resp2 = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "user", "content": STRUCTURING_PROMPT.format(text=raw_text)}],
        temperature=0,
        response_format={"type": "json_object"},
    )
    data = json.loads(resp2.choices[0].message.content)
    data["source_file"] = os.path.basename(image_path)
    data["raw_text"] = raw_text

    return data


print("파이프라인 함수 정의 완료!")

## 4. 일괄 처리 실행

In [ ]:
results = []

for i, img_path in enumerate(samples, 1):
    name = os.path.basename(img_path)
    print(f"[{i}/{len(samples)}] {name}...")

    try:
        data = process_single_image(img_path)
        results.append(data)
        print(f"  → 완료 (병원: {data.get('hospital_name', '?')}, 환자: {data.get('patient_name', '?')})")
    except Exception as e:
        print(f"  → 실패: {e}")
        results.append({"source_file": name, "error": str(e)})

print(f"\n=== {len(results)}개 처리 완료 ===")

## 5. DataFrame으로 변환

구조화된 결과를 pandas DataFrame으로 만들어 표로 확인합니다.

In [ ]:
# raw_text는 너무 길어서 별도 컬럼으로 분리
display_cols = [k for k in EXTRACTION_FIELDS.keys()] + ["source_file"]

df = pd.DataFrame(results)

# 표시용 DataFrame (raw_text 제외)
df_display = df[[c for c in display_cols if c in df.columns]]

print(f"총 {len(df_display)}건의 EMR 데이터 추출 완료\n")
df_display

## 6. CSV 다운로드

In [ ]:
from google.colab import files

# CSV 저장
output_csv = "emr_extracted.csv"
df_display.to_csv(output_csv, index=False, encoding="utf-8-sig")

# Excel 저장
output_xlsx = "emr_extracted.xlsx"
df_display.to_excel(output_xlsx, index=False)

print(f"파일 저장 완료:")
print(f"  - {output_csv}")
print(f"  - {output_xlsx}")

# 다운로드
files.download(output_csv)
files.download(output_xlsx)

## 7. (선택) 전체 원본 텍스트 포함 버전 저장

In [ ]:
# 원본 텍스트 포함 전체 데이터 저장
full_output = "emr_extracted_full.json"
with open(full_output, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

files.download(full_output)
print(f"{full_output} 다운로드!")

## 8. (실습) 자신만의 이미지로 테스트

왼쪽 파일 탐색기에 이미지를 업로드하거나, 아래 셀에서 업로드할 수 있습니다.

In [ ]:
from google.colab import files as colab_files

print("EMR 스캔 이미지를 업로드하세요 (PNG, JPG)")
uploaded = colab_files.upload()

for filename in uploaded:
    print(f"\n처리 중: {filename}")
    data = process_single_image(filename)

    print(f"\n--- 추출 결과 ---")
    for k, v in data.items():
        if k != "raw_text":
            print(f"  {k}: {v}")

---

## 정리

### 이번 주에 만든 것
- 이미지 → 텍스트 → JSON → CSV 전체 파이프라인
- 폴더 단위 일괄 처리 기능
- CSV/Excel 내보내기

### 파이프라인 구조
```
이미지 입력
  ↓ GPT-5.4-nano (비전)
원본 텍스트
  ↓ GPT-5.4-nano (structured output)
구조화 JSON
  ↓ pandas
DataFrame → CSV/Excel
```

### 다음 주 예고
LLM이 **스스로 도구를 사용**하게 만드는 AI Agent를 배웁니다.
예: "이 차트에서 진단코드를 찾아서 건강보험 청구 가능 여부를 확인해줘"  
→ LLM이 OCR → 코드 검색 → 판단까지 자동으로 수행